In [1]:
import pickle
import time
import tiktoken
import datetime
import json
import requests
import traceback
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from jsonschema import validate
from openai import OpenAI, RateLimitError, APITimeoutError, InternalServerError, Timeout
from tenacity import retry, stop_after_attempt, wait_incrementing, retry_if_exception_type, after_log, before_sleep_log
import logging
from IPython.display import clear_output

In [2]:
# enter your OpenRouter API key below
OPENROUTER_API_KEY = 'sk-...' 

In [3]:
request_limit_per_minute = 500
token_limit_per_minute = 2e6

request_timeout_seconds = 120   # maximum wait time for openAI to respond before triggering request timeout 
request_max_retries = 1         # number to times to automatically retry failed requests
tpm_wait_polling_seconds = 10    # if our internal TPM estimate thinks TPM limit is exceeded, how often to check if limit cleared

# global logger for static classes
logger = logging.getLogger()
logging.basicConfig(level=logging.INFO)

# Helper Class for Calling LLMs

In [4]:
class OpenRouter:
    def __init__(self, model_provider_order,
                 halt_on_error=True,
                 is_verbose=True,
                 timeout=request_timeout_seconds,
                 max_retries=request_max_retries,
                 request_limit_per_minute=request_limit_per_minute,
                 token_limit_per_minute=token_limit_per_minute,
                 tpm_wait_polling_seconds=tpm_wait_polling_seconds,
                 logger=logger,
                 api_key=OPENROUTER_API_KEY):
        self.model, self.provider_order, self.model_canonical_name = model_provider_order
        self.halt_on_error = halt_on_error
        self.is_verbose = is_verbose
        self.tpm_wait_polling_seconds = tpm_wait_polling_seconds
        self.request_limit_per_minute = request_limit_per_minute
        self.request_delay_seconds = 60.0 / request_limit_per_minute
        self.token_limit_per_minute = token_limit_per_minute
        self.response_history = []
        self.message_history = {}
        self.logger = logger
        likert_options = [
            "Very Inaccurate",
            "Moderately Inaccurate",
            "Neither Accurate nor Inaccurate",
            "Moderately Accurate",
            "Very Accurate",
        ]
        # Sort by length to match longer options (e.g., "Strongly Agree") before shorter ones (e.g., "Agree")
        self.likert_options = sorted(likert_options, key=len, reverse=True)
        self.default_seed = 1 if 'x-ai' in self.model else 0
        
        self.client = OpenAI(base_url="https://openrouter.ai/api/v1",
                             api_key = api_key,
                             timeout=timeout,
                             max_retries=max_retries)

        
    def extract_float_response(self, content):
        # Search for a float or integer between 0 and 100
        match = re.search(r'\b(100(?:\.0+)?|[0-9]?[0-9](?:\.\d+)?|\d{1,2})\b', content)
        if match:
            value = float(match.group(0))
            if 0.0 <= value <= 100.0:
                return json.dumps({"response": value})
        raise Exception("No valid float response found in: ", content)
    
    
    def get_running_cost_num_prompt_completion_tokens(self):
        """
        This function computes the total cost (estimated) of all
        messages sent by the instance of ChatGPT called from
        Returns: total_running_cost, total_num_prompt_tokens, total_num_response_tokens
        """
        n_prompt_tokens = np.sum([x.usage.prompt_tokens for x in self.response_history])
        n_completion_tokens = np.sum([x.usage.completion_tokens for x in self.response_history])
        total_cost = sum(r.usage.cost for r in self.response_history)
        return (total_cost,
                n_prompt_tokens,
                n_completion_tokens)

    def get_key_usage_credits(self):
        # get what OpenRouter says the api key has used in total
        # returns usage, total credits available
        resp = requests.get(
            "https://openrouter.ai/api/v1/credits",
            headers={"Authorization": f"Bearer {self.client.api_key}"}
        )
        resp.raise_for_status()
        info = resp.json()["data"]
        return info["total_usage"], info["total_credits"]

    # retry failing requests starting with 10 second wait,
    # increasing wait time by 10 seconds each retry, up to a max window of 120s (or 5 times)
    # the goal is to try to avoid hitting backoff,
    # we treat this as a last resort because of its runtime cost
    @retry(wait=wait_incrementing(start=10, increment=10, max=120),
           stop=stop_after_attempt(5),
           retry=retry_if_exception_type((RateLimitError, APITimeoutError, InternalServerError, Timeout)),
           before_sleep=before_sleep_log(logger, logging.INFO),
           after=after_log(logger, logging.INFO))
    def completion_with_backoff(self, client, **kwargs):
        return client.chat.completions.create(**kwargs)


    def check_internal_TPM_tracker(self, n_message_tokens):
        """
        Checks internal TPM count to see if a message with length = n_message_tokens
        can be sent. If not, it waits (sleeps - blocking) until the message delivery
        meets into TPM limit
        """
        now = datetime.datetime.now()
        one_minute_ago = now + datetime.timedelta(seconds=-60)
        self.token_count_history = [x for x in self.token_count_history if x[1] > one_minute_ago]
        n_tokens_past_minute = np.sum([x[0] for x in self.token_count_history]) + n_message_tokens
        # fixed delay waiting if TPM exceeded over past minute
        # this is cpu polling, so it doesnt cost money or much compute
        while n_tokens_past_minute > self.token_limit_per_minute:
            if self.is_verbose: self.logger.info(f'Internal TPM limit exceeded, waiting for {self.tpm_wait_polling_seconds} seconds...')
            time.sleep(self.tpm_wait_polling_seconds)
            now = datetime.datetime.now()
            one_minute_ago = now + datetime.timedelta(seconds=-60)
            self.token_count_history = [x for x in self.token_count_history if x[1] > one_minute_ago]
            n_tokens_past_minute = np.sum([x[0] for x in self.token_count_history]) + n_message_tokens
        now = datetime.datetime.now()
        self.token_count_history.append((n_message_tokens, now))


    def send_message(self, system_role, message, json_schema, validate_response=True):
        """
        This is the primary function used to send messages to GPT and get responses
        Steps are:
          - check that json schema meets our basic requirements
          - handle RPM and TPM limits as best as we can
            (when openai rejects requests for exceeding limits its much slower)
          - build and send the message using openai ChatCompletion api
          - perform basic validation on GPT's response
        This function either returns a ChatCompletion response object or None (if failure occurred)
        Errors are propogated using raised Exceptions
        """
        # sleep based on RPM limit (lazy logic, avoids keeping running count of actual requests per minute)
        time.sleep(self.request_delay_seconds)

        # check TPM limit (not lazy, uses running count of tokens per minute)
        try:
            encoding = tiktoken.encoding_for_model(self.model)
        except:
            encoding = tiktoken.encoding_for_model('gpt-4')
        n_message_tokens = len(encoding.encode(system_role)) + len(encoding.encode(message))
        self.logger.info(f'processing message with {n_message_tokens} tokens...')
        if n_message_tokens > self.token_limit_per_minute:
            return self.bad_response_output(f'Unable to send message as it exceeds TPM. Number of tokens in message = {n_message_tokens}')
        
        # build and send message over openai-api
        message_id = len(self.message_history.keys())
        self.message_history[message_id] = [] if 'x-ai' in self.model else [{"role": "system", "content": system_role}]
        self.message_history[message_id].append({"role": "user", "content": message})
        try:
            response = self.completion_with_backoff(self.client,
                                                    model=self.model,
                                                    messages=self.message_history[message_id],
                                                    temperature=0,
                                                    stream=False,
                                                    extra_body={"usage": {"include": True},
                                                                "reasoning": {# One of the following (not both):
                                                                              "effort": "medium", # Can be "high", "medium", or "low" (OpenAI-style)
                                                                              # Optional: Default is false. All models support this.
                                                                              "exclude": False # Set to true to exclude reasoning tokens from response
                                                                              },
                                                                "provider": {"order": self.provider_order, 
                                                                             "sort": "price",
                                                                             "data_collection": "deny",
                                                                             "allow_fallbacks": False}},
                                                    response_format={"type": "json_schema",
                                                                     "json_schema": json_schema},
                                                    seed=self.default_seed, logprobs=False)
            
            self.response_history.append(response)
            # reasoning models dont return a content field
            if response.choices[0].message.content is None:
                self.bad_response_output(f'None in message content')
                return None
            elif response.choices[0].message.content == '':
                if hasattr(response.choices[0].message, 'reasoning'):
                    if response.choices[0].message.reasoning != '':
                        response.choices[0].message.content = response.choices[0].message.reasoning
            else:
                pass
        except Exception as e:
            if self.halt_on_error:
                raise
            else:
                if self.is_verbose:
                    str_e = str(e)
                    self.logger.info(f'An exception occurred: {str_e}')
                    self.logger.info(traceback.format_exc())
                return None

        return (response, message_id)
        

    def bad_response_output(self, error):
        # general function for informing the user when an error occurs
        if self.halt_on_error:
            raise Exception(error)
        else:
            if self.is_verbose:
                self.logger.info(f'Error - {error}')
        return None

# Load AAPECS Data

In [8]:
DATA_ROOT = './Data'  # please contact the authors for access to the data
pheno_df, sub_transcripts, sub_dpds = pickle.load(open(f'{DATA_ROOT}/AAPECS.pickle', 'rb'))

# Prepare Prompt Template

In [9]:
questions_list = [
    {'text': 'My mood was up and down.','domain':'','qid':'dpds1'},
    {'text': 'I had frequent mood swings.','domain':'','qid':'dpds2'},
    {'text': 'My emotions were like a roller coaster.','domain':'','qid':'dpds3'},
    #{'text': 'I lost my temper.','domain':'','qid':'dpds4'},
    #{'text': 'I was easily irritated.','domain':'','qid':'dpds5'},
    #{'text': 'I felt overwhelmed with rage.','domain':'','qid':'dpds6'},
    {'text': 'I was not interested in doing much of anything.','domain':'','qid':'dpds7'},
    {'text': 'Nothing seemed to make me feel good.','domain':'','qid':'dpds8'},
    {'text': 'I get little to no pleasure out of things.','domain':'','qid':'dpds9'},
    {'text': 'I felt anxious.','domain':'','qid':'dpds10'},
    {'text': 'I worried about several things.','domain':'','qid':'dpds11'},
    {'text': 'I felt tense.','domain':'','qid':'dpds12'},
    #{'text': 'I didn\'t bother to think how my actions affected others.','domain':'','qid':'dpds13'},
    #{'text': 'I was indifferent to how others felt.','domain':'','qid':'dpds14'},
    #{'text': 'I couldn\'t be bothered with others\' feelings.','domain':'','qid':'dpds15'},
    {'text': 'I felt depressed.','domain':'','qid':'dpds16'},
    {'text': 'I felt hopeless.','domain':'','qid':'dpds17'},
    {'text': 'I generally focused on the negative side of things.','domain':'','qid':'dpds18'},
    #{'text': 'I insisted things be done my way.','domain':'','qid':'dpds19'},
    #{'text': 'I bossed people around.','domain':'','qid':'dpds20'},
    #{'text': 'I enjoyed having authority over others.','domain':'','qid':'dpds21'},
    {'text': 'I had difficulty expressing my feelings.','domain':'','qid':'dpds22'},
    {'text': 'I kept my emotions to myself.','domain':'','qid':'dpds23'},
    {'text': 'I had difficulty showing affection.','domain':'','qid':'dpds24'},
    #{'text': 'I wanted people to notice my body.','domain':'','qid':'dpds25'},
    #{'text': 'I wanted people to notice my talents.','domain':'','qid':'dpds26'},
    #{'text': 'I showed off when I got the chance.','domain':'','qid':'dpds27'},
    #{'text': 'I believed I was better than the people around me.','domain':'','qid':'dpds28'},
    #{'text': 'I was surrounded by people that were beneath me.','domain':'','qid':'dpds29'},
    #{'text': 'I felt like I deserved special treatment from others.','domain':'','qid':'dpds30'},
    {'text': 'I acted aggressively toward someone.','domain':'','qid':'dpds31'},
    {'text': 'I felt like I wanted to hurt someone.','domain':'','qid':'dpds32'},
    {'text': 'I thought about getting back at/getting even with someone.','domain':'','qid':'dpds33'},
    {'text': 'I behaved irresponsibly.','domain':'','qid':'dpds34'},
    {'text': 'I neglected my responsibilities.','domain':'','qid':'dpds35'},
    {'text': 'I let someone down.','domain':'','qid':'dpds36'},
    #{'text': 'I took advantage of someone.','domain':'','qid':'dpds37'},
    #{'text': 'I lied to someone.','domain':'','qid':'dpds38'},
    #{'text': 'I used others for my own gain.','domain':'','qid':'dpds39'},
    #{'text': 'I was suspicious of others.','domain':'','qid':'dpds40'},
    #{'text': 'I felt like others were trying to take advantage of me.','domain':'','qid':'dpds41'},
    #{'text': 'I suspected that someone lied to me.','domain':'','qid':'dpds42'},
    #{'text': 'I lost interest in the tasks I started.','domain':'','qid':'dpds43'},
    #{'text': 'I quit tasks as soon as I got bored.','domain':'','qid':'dpds44'},
    #{'text': 'I was easily distracted.','domain':'','qid':'dpds45'},
    {'text': 'I did something on impulse.','domain':'','qid':'dpds46'},
    {'text': 'I did things without worrying about the consequences.','domain':'','qid':'dpds47'},
    {'text': 'I acted without planning.','domain':'','qid':'dpds48'},
    #{'text': 'I did something that could have gotten me in trouble.','domain':'','qid':'dpds49'},
    #{'text': 'I got in trouble for something I did.','domain':'','qid':'dpds50'},
    #{'text': 'I broke the rules.','domain':'','qid':'dpds51'},
    {'text': 'I made sure everything I did was perfect.','domain':'','qid':'dpds52'},
    {'text': 'I strived to be flawless.','domain':'','qid':'dpds53'},
    {'text': 'I set high standards for myself and others.','domain':'','qid':'dpds54'},
    {'text': 'I worried about being abandoned.','domain':'','qid':'dpds55'},
    {'text': 'I thought that others might not like me.','domain':'','qid':'dpds56'},
    {'text': 'I was worried people were pulling away from me.','domain':'','qid':'dpds57'},
    {'text': 'I did something dangerous just for the thrill.','domain':'','qid':'dpds58'},
    {'text': 'Avoided a situation that would have put me at risk.','domain':'','qid':'dpds59'},
    {'text': 'I took a risk.','domain':'','qid':'dpds60'},
    #{'text': 'I thought about sex.','domain':'','qid':'dpds61'},
    #{'text': 'I wasn\'t interested in sex or romance.','domain':'','qid':'dpds62'},
    #{'text': 'I had a strong desire for romance or sex.','domain':'','qid':'dpds63'},
    {'text': 'I said something offensive to someone.','domain':'','qid':'dpds64'},
    {'text': 'I insulted someone.','domain':'','qid':'dpds65'},
    {'text': 'I made a fool of someone.','domain':'','qid':'dpds66'},
    #{'text': 'I thought about injuring/harming myself on purpose.','domain':'','qid':'dpds67'},
    #{'text': 'I did something to injure myself on purpose.','domain':'','qid':'dpds68'},
    #{'text': 'I had the urge to hurt myself.','domain':'','qid':'dpds69'},
    {'text': 'I didn\'t want to be around others.','domain':'','qid':'dpds70'},
    {'text': 'I kept to myself.','domain':'','qid':'dpds71'},
    {'text': 'I avoided other people.','domain':'','qid':'dpds72'},
    #{'text': 'I let others tell me what to do.','domain':'','qid':'dpds73'},
    #{'text': 'I let others make decisions for me.','domain':'','qid':'dpds74'},
    #{'text': 'I let others take advantage of me.','domain':'','qid':'dpds75'},
    {'text': 'I put work above everything else.','domain':'','qid':'dpds76'},
    {'text': 'I worked longer than others around me.','domain':'','qid':'dpds77'},
    {'text': 'I didn’t leave any time for relaxing or fun.','domain':'','qid':'dpds78'},
    #{'text': 'I acted on my emotions.','domain':'','qid':'dpds79'},
    #{'text': 'I acted on impulse while feeling upset.','domain':'','qid':'dpds80'},
    #{'text': 'I had a hard time controlling my urges when stressed.','domain':'','qid':'dpds81'}
]

for i in range(len(questions_list)):
    questions_list[i]['text'] = f'Question {i+1}: ' + questions_list[i]['text']

In [13]:
def get_prompt(thoughts, question):
    prompt_template = f"""
Your task is to respond to the following question based on the participant's daily diaries of the most significant event that occurred during the day, provided below. Respond as though you are the individual who generated these thoughts, reflecting their personality traits.
Base your answer on inferred personality traits. Think carefully about what the thoughts imply about tendencies and behaviors.

Question to answer:
{question}

Participant's daily diaries:
{thoughts}

Respond with a single number from 0 to 100, where:
- 0 means "not at all"
- 100 means "very much"

Return only the number. Do not include any explanation or punctuation.
"""
    return prompt_template

In [ ]:
system_role = ''

In [14]:
def generic_messaging_wrapper(chat, system_role, message):
    # with halt_on_error set to True in ChatGPT class, 
    # we use exception propogation to handle errors and edge-cases
    message_history_id = -1
    required_values = None
    try:
        response, message_history_id = chat.send_message(system_role=system_role,
                                                         message=message)
        assert response is not None
        response_json = json.loads(response.choices[0].message.content)
        required_values = response_json['response']
    except Exception as e:
        response_json = {}
        chat.logger.info(f'Messaging wrapper failure - {str(e)}')
        print(traceback.format_exc())

    cost, n_prompt_tokens, n_completion_tokens = chat.get_running_cost_num_prompt_completion_tokens()
    return required_values, cost

# Multiproc LLM Calling Functions

In [17]:
from multiprocessing import Process, Manager

def multiproc_neo_wrapper(thoughts, question_i, name, return_dict, model_to_evaluate):
    try:
        chat = ChatGPT(model_provider_order=model_to_evaluate)
        dim_question = questions_list[question_i]['text']
        message = get_prompt(thoughts, dim_question)
        required_values, cost = generic_messaging_wrapper(chat, system_role, message)
        return_dict[f'{name}'] = (required_values, cost)
    except Exception as e:
        pass  # silent failures 

In [52]:
return_dict = Manager().dict()

In [19]:
def run_until_resolved(good_th_dict, questions, return_dict, model_to_evaluate,
                       max_attempts=5, sleep_between=0.1, max_concurrent_calls=8100):
    n_subs = len(good_th_dict)
#    total_requests = np.sum([len(good_th_dict[x]) for x in good_th_dict]) * len(questions)
    total_requests = len(good_th_dict) * len(questions)
    start_time = time.time()
    
    def format_time(seconds):
        mins, secs = divmod(int(seconds), 60)
        hrs, mins = divmod(mins, 60)
        return f"{hrs:02d}:{mins:02d}:{secs:02d}"
    
    for attempt in range(1, max_attempts + 1):
        procs = []
        n_missing = 0
        missing_items = []

        # Check for missing entries
        for subject in good_th_dict:
            for question_i in range(len(questions)):
                name = f'{subject}-{question_i}'
                if name not in return_dict:
                    n_missing += 1
                    missing_items.append((subject, question_i, name))
                else:
                    if return_dict[name][0] is None:
                        n_missing += 1
                        missing_items.append((subject, question_i, name))

        if n_missing == 0:
            print("All responses successfully completed.")
            break
        else:
            print(f"Attempt {attempt}: {n_missing} / {total_requests} requests missing.")

        # Dispatch missing jobs
        for (subject_i, (subject, question_i, name)) in enumerate(missing_items):
            transcript = "\n".join(sub_transcripts[subject])
            proc = Process(target=multiproc_neo_wrapper, args=(transcript, question_i, name, return_dict, model_to_evaluate))
            proc.start()
            procs.append(proc)
            time.sleep(sleep_between)
            clear_output(wait=True)

            if len(procs) >= max_concurrent_calls:
                for proc in procs:
                    proc.join()
                procs = []
            
                # Intermediate update
                completed_items = dict(return_dict)
                running_cost = np.sum([completed_items[x][-1] for x in completed_items])
                n_completed = len(completed_items)
                avg_cost = running_cost / n_completed if n_completed else 0
                estimated_total_cost = avg_cost * total_requests

                elapsed_time = time.time() - start_time
                avg_time_per_call = elapsed_time / n_completed if n_completed else 0
                remaining_calls = total_requests - n_completed
                estimated_time_remaining = avg_time_per_call * remaining_calls

                clear_output(wait=True)
                print(f"Attempt {attempt}")
                print(f"Completed: {n_completed}/{total_requests}")
                print(f"Running cost: ${running_cost:.3f}")
                print(f"Estimated total cost: ${estimated_total_cost:.3f}")
                print(f"Elapsed time: {format_time(elapsed_time)}")
                print(f"Estimated time remaining: {format_time(estimated_time_remaining)}")
        clear_output(wait=True)    
        # Final join
        for proc in procs:
            proc.join()

        running_cost = np.sum([return_dict[x][-1] for x in return_dict])
        clear_output(wait=True)
        print(f"Total running cost: {running_cost:.3f}")
        

        clear_output(wait=True)
        n_missing = 0
        # Check for missing entries
        for subject in good_th_dict:
            for question_i in range(len(questions)):
                name = f'{subject}-{question_i}'
                if name not in return_dict:
                    n_missing += 1
        completed_dict = dict(return_dict)
        completed_items = [v for v in completed_dict.values()]
        total_cost = np.sum([x[-1] for x in completed_items])
        n_completed = len(completed_items)
        avg_cost = total_cost / n_completed if n_completed else 0
        total_elapsed_time = time.time() - start_time

        print(f"Total responses expected: {total_requests}")
        print(f"Successful: {n_completed}")
        print(f"Failed: {n_missing}")
        print(f"Total cost: ${total_cost:.3f}")
        print(f"Avg cost per response: ${avg_cost:.4f}")
        print(f"Total runtime: {format_time(total_elapsed_time)}")
        
    else:
        n_missing = 0
        # Check for missing entries
        for subject in good_th_dict:
            for question_i in range(len(questions)):
                name = f'{subject}-{question_i}'
                if name not in return_dict:
                    n_missing += 1
                else:
                    if return_dict[name][0] is None:
                        n_missing += 1
                        missing_items.append((subject, question_i, name))

        print(f'!!! Max attempts reached. {n_missing} requests are still unresolved. !!!')

# Run and Save Outputs

In [20]:
llama_4_maverick = ('meta-llama/llama-4-maverick', ['Fireworks', 'Together', 'Kluster'], 'llama_maverick')
gemini_flash = ('google/gemini-2.5-flash', ['Google', 'Google AI Studio'], 'gemini_flash')
qwen_235B = ('qwen/qwen3-235b-a22b', ['DeepInfra', 'Kluster', 'Parasail', 'Together', 'Nebius'], 'qwen3_235B')  
gpt_41 = ('openai/gpt-4.1', ['OpenAI'], 'gpt_41')
gpt_41_mini = ('openai/gpt-4.1-mini', ['OpenAI'], 'gpt_41_mini')
claude_sonnet = ('anthropic/claude-3.7-sonnet', ['Anthropic', 'Amazon Bedrock', 'Google', 'Google AI Studio'], 'claude_sonnet')
grok_3 = ('x-ai/grok-3', ['xAI'], 'grok3')
grok_3_mini = ('x-ai/grok-3-mini',['xAI'],'grok3mini')
gpt_oss = ('openai/gpt-oss-20b',['OpenAI','nCompass','DeepInfra','Together'],'gpt_oss_20b')
gpt_5 = ('openai/gpt-5',['OpenAI'],'gpt_5')
gpt_5_mini = ('openai/gpt-5-mini',['OpenAI'],'gpt_5_mini')
gpt_5_nano = ('openai/gpt-5-nano',['OpenAI'],'gpt_5_nano')
claude_sonnet_4 = ('anthropic/claude-sonnet-4',['Anthropic','Amazon Bedrock','Google'],'claude_sonnet_4')
qwen3_next = ('qwen/qwen3-next-80b-a3b-instruct',['DeepInfra','nCompass','Parasail','NovitaAI','GMICloud'],'qwen3_next')
qwen3_30b = ('qwen/qwen3-30b-a3b-instruct-2507',['SiliconFlow','Nebius','Alibaba'],'qwen3_30b')

In [21]:
models = [claude_sonnet_4, gpt_5, grok_3, llama_4_maverick, gemini_flash, qwen_235B]

In [ ]:
for MODEL_TO_EVALUATE in models:
    return_dict = Manager().dict()
    run_until_resolved(sub_transcripts, questions_list, return_dict, MODEL_TO_EVALUATE)
    model_canonical_name = MODEL_TO_EVALUATE[2]
    pickle.dump(dict(return_dict), open(f'./Results/return_dict_{model_canonical_name}_per_subject.pickle', 'wb'))
    rows = []
    for subject in sub_transcripts:         
        row = {
            'subject': subject,
        }
        for question_i in range(len(questions_list)):
            k = f'{subject}-{question_i}'
            val = return_dict[k][0] if k in return_dict else float('nan')
            row[f"{questions_list[question_i]['qid']}_llm"] = val
            val = np.mean(sub_dpds[k]) if k in sub_dpds else float('nan')
            row[f"{questions_list[question_i]['qid']}"] = val

        rows.append(row)

    out_df = pd.DataFrame(rows)

    model_canonical_name = MODEL_TO_EVALUATE[2]
    out_df.to_csv(f'./Results/dpds_{model_canonical_name}_per_subject_responses.csv')